#### Using data olist from kaggle dataset "olistbr/brazilian-ecommerce"

### Work directory: projects/DE/olist-analysis
### Environment

Activate project virtual environment in zsh/bash

``` source .venv/bin/activate ```

Check interpreter

``` which python ```

In [1]:
import sys

print(sys.executable)

/Users/zhw/projects/DE/olist-analysis/.venv/bin/python


#### import python package, load data path

In [5]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw"


In [ ]:
# customers = pd.read_csv(os.path.join(data_path, "olist_customers_dataset.csv"))

csv_files = sorted(DATA_PATH.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found in {DATA_PATH}")

# tables = {}
# for csv_file in csv_files:
#     tables[csv_file.stem] = pd.read_csv(csv_file)

tables = {
    csv_file.stem: pd.read_csv(csv_file)
    for csv_file in csv_files
}

tables.keys()

dict_keys(['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_orders_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation'])

#### now load all data table

In [ ]:
table_summary = []

for table_name, df in tables.items():
    table_summary.append({
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
    })

# table_summary = pd.DataFrame(table_summary)
# table_summary = table_summary.sort_values("rows", ascending=False)
# table_summary = table_summary.reset_index(drop=True)

table_summary = (
    pd.DataFrame(table_summary)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

table_summary

,table,rows,columns,duplicate_rows,missing_values,memory_mb
0,olist_geolocation_dataset,1000163,5,261831,0,130.264880
1,olist_order_items_dataset,112650,7,0,0,35.989649
2,olist_order_payments_dataset,103886,5,0,0,16.229413
3,olist_customers_dataset,99441,5,0,0,26.586405
4,olist_orders_dataset,99441,8,0,4908,52.937277
5,olist_order_reviews_dataset,99224,7,0,145903,39.124777
6,olist_products_dataset,32951,9,0,2448,6.296564
7,olist_sellers_dataset,3095,4,0,0,0.588103
8,product_category_name_translation,71,2,0,0,0.008999


In [ ]:
table_summary = pd.DataFrame([
    {
        "table": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "missing_values": df.isna().sum().sum(),
        "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
    }
    for table_name, df in tables.items()
]).sort_values("rows", ascending=False).reset_index(drop=True)

table_summary

In [8]:
customers = tables["olist_customers_dataset"]
geolocation = tables["olist_geolocation_dataset"]
orders = tables["olist_orders_dataset"]
order_items = tables["olist_order_items_dataset"]
order_payments = tables["olist_order_payments_dataset"]
order_reviews = tables["olist_order_reviews_dataset"]
products = tables["olist_products_dataset"]
sellers = tables["olist_sellers_dataset"]
product_category_name_translation = tables["product_category_name_translation"]

#### next check the data structure

In [25]:
for table_name, df in tables.items():
    print("="*50)
    print(f"Table: {table_name}") # f-string allow variable interpolation
    print(df.shape)
    # print(df.head())
    df.info()
    print(f"null values: \n{df.isna().sum()}")

Table: olist_customers_dataset
(99441, 5)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB
null values: 
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64
Table: olist_geolocation_dataset
(1000163, 5)
<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  


In [ ]:
## Column Profiling

column_profile = pd.DataFrame([
    {
        "table": table_name,
        "column": col,
        "dtype": str(df[col].dtype),
        "rows": df.shape[0],
        "n_unique": df[col].nunique(),
        "null_count": df[col].isna().sum(),
        "null_pct": df[col].isna().mean() * 100,
        "sample_value": df[col].dropna().iloc[0] if df[col].notna().any() else None
    }
    for table_name, df in tables.items()
    for col in df.columns
])


In [105]:
column_profile[
    column_profile["table"] == "olist_order_items_dataset"
]

,table,column,dtype,rows,n_unique,null_count,null_pct,sample_value
10,olist_order_items_dataset,order_id,str,112650,98666,0,0.0,00010242fe8c5a6d1ba2dd792cb16214
11,olist_order_items_dataset,order_item_id,int64,112650,21,0,0.0,1
12,olist_order_items_dataset,product_id,str,112650,32951,0,0.0,4244733e06e7ecb4970a6e2683c13e61
13,olist_order_items_dataset,seller_id,str,112650,3095,0,0.0,48436dade18ac8b2bce089ec2a041202
14,olist_order_items_dataset,shipping_limit_date,str,112650,93318,0,0.0,2017-09-19 09:45:35
15,olist_order_items_dataset,price,float64,112650,5968,0,0.0,58.9
16,olist_order_items_dataset,freight_value,float64,112650,6999,0,0.0,13.29


In [96]:
sellers.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [77]:
customers["customer_zip_code_prefix"].value_counts()

customer_zip_code_prefix
22790    142
24220    124
22793    121
24230    117
22775    110
        ... 
87145      1
98860      1
5538       1
74980      1
99043      1
Name: count, Length: 14994, dtype: int64

## Table Grain

Grain defines the real-world meaning of one row.

| Table | Grain / One row represents |
|---|---|
| `olist_customers_dataset` | One customer record |
| `olist_geolocation_dataset` | One geolocation record for a ZIP code prefix |
| `olist_orders_dataset` | One order |
| `olist_order_items_dataset` | One item record within an order |
| `olist_order_payments_dataset` | One payment record for an order |
| `olist_order_reviews_dataset` | One review record for an order |
| `olist_products_dataset` | One product |
| `olist_sellers_dataset` | One seller |
| `product_category_name_translation` | One product-category name translation mapping |

## Candidate Key Assumptions

Candidate keys are fields or combinations of fields that are expected to uniquely identify one row according to the table grain.

Superkey = any set of columns that uniquely identifies a row

Candidate key = a minimal superkey

| Table | Candidate Key Assumption | Reason |
|---|---|---|
| `olist_customers_dataset` | `customer_id` | Each row represents one customer record, and `customer_id` is expected to identify that record uniquely. |
| `olist_geolocation_dataset` | None identified | ZIP code prefixes are not unique, and multiple geolocation records may exist for the same prefix. |
| `olist_orders_dataset` | `order_id` | Each row represents one order. |
| `olist_order_items_dataset` | (`order_id`, `order_item_id`) | An order can contain multiple item records, so the item sequence is needed within each order. |
| `olist_order_payments_dataset` | (`order_id`, `payment_sequential`) | An order can have multiple payment records. |
| `olist_order_reviews_dataset` | `review_id` | Each row represents a review record, so `review_id` is expected to identify the review uniquely. |
| `olist_products_dataset` | `product_id` | Each row represents one product. |
| `olist_sellers_dataset` | `seller_id` | Each row represents one seller. |
| `product_category_name_translation` | `product_category_name` | Each source category is expected to have one English translation mapping. |

In [ ]:
## Candidate Key Validation

